# UPF Demo

This python notebook shows the basic usage of the UPF planning library.

## Setup the library and the planners

We start by downloading (from github) the UPF library and the two planners we currently have at our disposal, namely `pyperplan` and `tamer`.

First, we download the upf and install its dependencies

In [ ]:
!rm -rf upf && git clone https://github.com/aiplan4eu/upf && cd upf && pip install -r requirements.txt

Cloning into 'upf'...
remote: Enumerating objects: 3423, done.
remote: Counting objects: 100% (2798/2798), done.
remote: Compressing objects: 100% (1571/1571), done.
remote: Total 3423 (delta 2105), reused 1823 (delta 1224), pack-reused 625
Receiving objects: 100% (3423/3423), 652.21 KiB | 4.26 MiB/s, done.
Resolving deltas: 100% (2506/2506), done.
  Cloning https://github.com/aig-upf/tarski.git (to revision ebfda1c13ac908904d5b74587971cc7149e73d85) to /tmp/pip-install-jcsva9f2/tarski_1d9ce844c8c24634921f6d9232d299e5
  Running command git clone -q https://github.com/aig-upf/tarski.git /tmp/pip-install-jcsva9f2/tarski_1d9ce844c8c24634921f6d9232d299e5
  Running command git rev-parse -q --verify 'sha^ebfda1c13ac908904d5b74587971cc7149e73d85'
  Running command git fetch -q https://github.com/aig-upf/tarski.git ebfda1c13ac908904d5b74587971cc7149e73d85
  Running command git checkout -q ebfda1c13ac908904d5b74587971cc7149e73d85
  Installing build dependencies ... done
  Getting requirements to

Then, we download and install pyperplan

In [ ]:
!rm -rf pyperplan-upf && git clone https://github.com/aiplan4eu/pyperplan-upf && cd pyperplan-upf/ && git checkout temporary-branch && bash install.sh

Cloning into 'pyperplan-upf'...
remote: Enumerating objects: 227, done.
remote: Counting objects: 100% (227/227), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 227 (delta 124), reused 191 (delta 92), pack-reused 0
Receiving objects: 100% (227/227), 38.87 KiB | 7.77 MiB/s, done.
Resolving deltas: 100% (124/124), done.
Branch 'temporary-branch' set up to track remote branch 'temporary-branch' from 'origin'.
Switched to a new branch 'temporary-branch'
     |████████████████████████████████| 69 kB 5.2 MB/s 
pyperplan installed successfully!
upf_pyperplan installed successfully!


Finally, we download and install tamer

In [ ]:
!rm -rf tamer-upf && git clone https://github.com/aiplan4eu/tamer-upf && cd tamer-upf && git checkout temporary-branch && bash install.sh

Cloning into 'tamer-upf'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 205 (delta 132), reused 133 (delta 68), pack-reused 0
Receiving objects: 100% (205/205), 50.24 KiB | 2.51 MiB/s, done.
Resolving deltas: 100% (132/132), done.
Branch 'temporary-branch' set up to track remote branch 'temporary-branch' from 'origin'.
Switched to a new branch 'temporary-branch'
pytamer installed successfully!
upf_tamer installed successfully!


In order to use the UPF, we indicate to python where to find the library by extending the search path.

In [ ]:
import sys
sys.path.append('upf/')

We are now ready to use the UPF!

## UPF Demo

### Basic imports
The basic imports we need for this demo are abstracted in the `shortcuts` package. Moreover we import the PDDL input/output modules.

In [ ]:
import upf
from upf.shortcuts import *
from upf.io.pddl_writer import PDDLWriter
from upf.io.pddl_reader import PDDLReader

### Problem definition via code

In this example, we will model a very simple robot navigation problem.

#### Types

The first thing to do is to introduce a "UserType" to model the concept of a location. It is possible to introduce as many types as needed; then, for each type we will define a set of objects of that type.  

In addition to `UserType`s we have three built-in types: `Bool`, `Real` and `Integer`. 

In [ ]:
Location = UserType('Location')

#### Fluents and constants

The basic variables of a planning problem are called "fluents" and are quantities that can change over time. Fluents can have differen types, in this first example we will stick to classical "predicates" that are fluents of boolean type. Moreover, fluents can have parameters: effectively describing multiple variables.

For example, a booean fluent `connected` with two parameters of type `Location` (that can be interpreted as `from` and `to`) can be used to model a graph of locations: there exists an edge between two locations `a` and `b` if `connected(a, b)` is true.

In this example, `connected` will be a constant (i.e. it will never change in any execution), but another fluent `robot_at` will be used to model where the robot is: the robot is in locatiopn `l` if and only if `robot_at(l)` is true (we will ensure that exactly one such `l` exists, so that the robot is always in one single location).

In [ ]:
robot_at = upf.model.Fluent('robot_at', BoolType(), [Location])
connected = upf.model.Fluent('connected', BoolType(), [Location, Location])

#### Actions

Now we have the problem variables, but in order to describe the possible evolutions of a systems we need to describe how these variables can be changed and how they can evolve. We model this problem using classica, action-based planning, where a set of actions is used to characterize the possible transitions of the system from a state to another.

An action is a transition that can be applied if a specified set of preconditions is satisfied and that prescribes a set of effects that change the value of some fluents. All the fluents that are subjected to the action effects are unchanged.

We allow _lifted_ actions, that are action with parameters: the parameters can be used to specify preconditions or effects and the planner will select among the possible values of each parameters the ones to be used to characterize a specific action. 

In our example, we introduce an action called `move` that has two parameters of type `Location` indicating the current position of the robot `l_from` and the intended destination of the movement `l_to`. The `move(a, b)` action is applicable only when the robot is in position `a` (i.e. `robot_at(a)`) and if `a` and `b` are connected locations (i.e. `connected(a, b)`). As a result of applying the action `move(a, b)`, the robot is no longer in `a` and is instead in location `b`.

In the UPF, we can create actions by instantiating the `upf.InstantaneousAction` class; parameters are specified as keyword arguments to the constructor as shown below. Preconditions and effects are added by means of the `add_precondition` and `add_effect` methods. 

In [ ]:
move = upf.model.InstantaneousAction('move', l_from=Location, l_to=Location)
l_from = move.parameter('l_from')
l_to = move.parameter('l_to')
move.add_precondition(connected(l_from, l_to))
move.add_precondition(robot_at(l_from))
move.add_effect(robot_at(l_from), False)
move.add_effect(robot_at(l_to), True)
print(move)

action move(Location l_from, Location l_to) {
    preconditions = [
      connected(l_from, l_to)
      robot_at(l_from)
    ]
    effects = [
      robot_at(l_from) := false
      robot_at(l_to) := true
    ]
  }


#### Creating the problem

The class that represents a planning problem is `upf.Problem`, it contains the set of fluents, the actions, the objects, an intial value for all the fluents and a goal to be reached by the planner. We start by adding the entities we created so far. Note that entities are not bound to one problem, we can create the actions and fluents one and create multiple problems with them.

In [ ]:
problem = upf.model.Problem('robot')
problem.add_fluent(robot_at, default_initial_value=False)
problem.add_fluent(connected, default_initial_value=False)
problem.add_action(move)

The set of objects is a set of `upf.Object` instances, each represnting an element of the domain. In this example, we create `NLOC` (set to 10) locations named `l0` to `l9`. We can create the set of objects and add it to the problem as follows.

In [ ]:
NLOC = 10
locations = [upf.model.Object('l%s' % i, Location) for i in range(NLOC)]
problem.add_objects(locations)

Then, we need to specify the initial state. We used the `default_initial_value` specification when adding the fluents, so it suffices to indicate the fluents that are initially true (this is called "small-world assumption". Without this specification, we would need to initialize all the possible instantiation of all the fluents).

In this example, we connect location `li` with location `li+1`, creating a simple "linear" graph lof locations and we set the initial position of the robot in location `l0`.

In [ ]:
problem.set_initial_value(robot_at(locations[0]), True)
for i in range(NLOC - 1):
    problem.set_initial_value(connected(locations[i], locations[i+1]), True)

Finally, we set the goal of the problem. In this example, we set ouselves to reach location `l9`.

In [ ]:
problem.add_goal(robot_at(locations[-1]))
print(problem)

problem name = robot

types = [Location]

fluents = [
  bool robot_at[Location]
  bool connected[Location, Location]
]

actions = [
  action move(Location l_from, Location l_to) {
    preconditions = [
      connected(l_from, l_to)
      robot_at(l_from)
    ]
    effects = [
      robot_at(l_from) := false
      robot_at(l_to) := true
    ]
  }
]

objects = [
  Location: [l0, l1, l2, l3, l4, l5, l6, l7, l8, l9]
]

initial values = [
  robot_at(l0) := true
  connected(l0, l1) := true
  connected(l1, l2) := true
  connected(l2, l3) := true
  connected(l3, l4) := true
  connected(l4, l5) := true
  connected(l5, l6) := true
  connected(l6, l7) := true
  connected(l7, l8) := true
  connected(l8, l9) := true
  robot_at(l1) := false
  robot_at(l2) := false
  robot_at(l3) := false
  robot_at(l4) := false
  robot_at(l5) := false
  robot_at(l6) := false
  robot_at(l7) := false
  robot_at(l8) := false
  robot_at(l9) := false
  connected(l0, l0) := false
  connected(l1, l0) := false
  connected(l

### Solving Planning Problems

The most direct way to solve a planning problem is to select an available planning engine by name and use it to solve the problem. In the following we use `pyperplan` to solve the problem and print the plan.

In [ ]:
with OneshotPlanner(name='pyperplan') as planner:
    plan = planner.solve(problem)
    print("Pyperplan returned: %s" % plan)

Pyperplan returned: [move(l0, l1), move(l1, l2), move(l2, l3), move(l3, l4), move(l4, l5), move(l5, l6), move(l6, l7), move(l7, l8), move(l8, l9)]


Unfortunately, at the moment our planner `tamer` does not run in the Google Colab environment (we compiled it against python 3.8, Colab uses 3.7), we will try to fix this issue as soon as possible. This is shown by the following error.

In [ ]:
import pytamer

ImportError: ignored

The UPF can also automatically select, among the available planners installed on the system, one that is expressive enough for the problem at hand.

In [ ]:
with OneshotPlanner(problem_kind=problem.kind()) as planner:
    plan = planner.solve(problem)
    print("%s returned: %s" % (planner.name(), plan))

Pyperplan returned: [move(l0, l1), move(l1, l2), move(l2, l3), move(l3, l4), move(l4, l5), move(l5, l6), move(l6, l7), move(l7, l8), move(l8, l9)]


In this example, Pyperplan was selected. The `problem.kind()` function, returns an object that describes the characteristics of the problem.

In [ ]:
print(problem.kind().features())

{'FLAT_TYPING'}


#### Beyond plan generation

`OneshotPlanner` is not the only operation mode we can invoke from the UPF, it is just one way to interact with a planning engine. Another useful functionality is `PlanValidation` that checks if a plan is valid for a problem.

In [ ]:
with PlanValidator(problem_kind=problem.kind()) as validator:
    if validator.validate(problem, plan):
        print('The plan is valid')
    else:
        print('The plan is invalid')

The plan is valid


#### Parallel planning

We can invoke different instances of a planner in parallel or different planners and return the first plan that is generated effortlessly.

In [ ]:
with OneshotPlanner(names=['tamer', 'tamer', 'pyperplan'],
                    params=[{'heuristic': 'hadd'}, {'heuristic': 'hmax'}, {}]) as planner:
    plan = planner.solve(problem)
    print("%s returned: %s" % (planner.name(), plan))

KeyError: ignored

### PDDL I/O

The upf allows to read and write PDDL problems effortlessly.

In [ ]:
w = PDDLWriter(problem)
print(w.get_domain())
print(w.get_problem())

(define (domain robot-domain)
 (:requirements :strips :typing)
 (:types Location)
 (:predicates (robot_at ?p0 - Location) (connected ?p0 - Location ?p1 - Location))
 (:action move
  :parameters ( ?l_from - Location ?l_to - Location)
  :precondition (and (connected ?l_from ?l_to) (robot_at ?l_from))
  :effect (and (not (robot_at ?l_from)) (robot_at ?l_to)))
)

(define (problem robot-problem)
 (:domain robot-domain)
 (:objects 
   l0 l1 l2 l3 l4 l5 l6 l7 l8 l9 - Location
 )
 (:init (robot_at l0) (connected l0 l1) (connected l1 l2) (connected l2 l3) (connected l3 l4) (connected l4 l5) (connected l5 l6) (connected l6 l7) (connected l7 l8) (connected l8 l9))
 (:goal (and (robot_at l9)))
)



In [ ]:
reader = PDDLReader()
pddl_problem = reader.parse_problem('upf/upf/test/pddl/depot/domain.pddl', 'upf/upf/test/pddl/depot/problem.pddl')
print(pddl_problem)

problem name = depotprob1818

types = [object]

fluents = [
  bool at[object, object]
  bool on[object, object]
  bool in[object, object]
  bool lifting[object, object]
  bool available[object]
  bool clear[object]
  bool place[object]
  bool locatable[object]
  bool depot[object]
  bool distributor[object]
  bool truck[object]
  bool hoist[object]
  bool surface[object]
  bool pallet[object]
  bool crate[object]
]

actions = [
  action drive(object ?x, object ?y, object ?z) {
    preconditions = [
      (truck(?x) and place(?y) and place(?z) and at(?x, ?y))
    ]
    effects = [
      at(?x, ?z) := true
      at(?x, ?y) := false
    ]
  }
  action lift(object ?x, object ?y, object ?z, object ?p) {
    preconditions = [
      (hoist(?x) and crate(?y) and surface(?z) and place(?p) and at(?x, ?p) and available(?x) and at(?y, ?p) and on(?y, ?z) and clear(?y))
    ]
    effects = [
      lifting(?x, ?y) := true
      clear(?z) := true
      at(?y, ?p) := false
      clear(?y) := false
    

A parsed PDDL problem is just a normal problem that can be solved.

In [ ]:
print(pddl_problem.kind().features())
with OneshotPlanner(name='pyperplan') as planner:
    plan = planner.solve(pddl_problem)
    print("%s returned: %s" % (planner.name(), plan))


{'FLAT_TYPING'}
Pyperplan returned: [lift(hoist1, crate0, pallet1, distributor0), lift(hoist0, crate1, pallet0, depot0), load(hoist0, crate1, truck1, depot0), drive(truck1, depot0, distributor0), load(hoist1, crate0, truck1, distributor0), unload(hoist1, crate1, truck1, distributor0), drive(truck1, distributor0, distributor1), drop(hoist1, crate1, pallet1, distributor0), unload(hoist2, crate0, truck1, distributor1), drop(hoist2, crate0, pallet2, distributor1)]
